In [66]:
from pdb import set_trace as st
from pprint import pprint
import json

import msgspec
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
import os
import re 

from concurrent.futures import ThreadPoolExecutor
from sutime import SUTime

global sutime, heideltime_parser
sutime = SUTime(mark_time_ranges=True, include_range=True)

from python_heideltime import Heideltime

heideltime_parser = Heideltime()
heideltime_parser.set_document_type("NEWS")

from lxml import etree
parser = etree.XMLParser()

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()

timex_text_pattern = re.compile(r"<TIMEX3[^>]*>(.*?)</TIMEX3>")
timex_value_pattern = re.compile(r'<TIMEX3[^>]*\bvalue="([^"]+)"')
timex_pattern = re.compile(r'<TIMEX3[^>]*\bvalue="([^"]+)"[^>]*>(.*?)</TIMEX3>')

[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Registering annotator sutime with class edu.stanford.nlp.time.TimeAnnotator
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator tokenize
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ssplit
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator pos
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator lemma
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ner
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator sutime


# Utils

In [67]:
def read_json(file_path, jsonl=False):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise ValueError("filepath is not a file")
    # if not file_path.suffix == ".jsonl" and jsonl:
    #     raise ValueError("the file is not jsonl")
    # if not file_path.suffix == ".json" and not jsonl:
    #     raise ValueError("the file is not json")
    file_path = file_path.__str__()

    with open(file_path, "rb") as file:
        data = file.read()
    if jsonl:
        output = decoder.decode_lines(data)
    else:
        output = decoder.decode(data)

    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output


def write_json(file_path, data, jsonl=False):
    with open(file_path, "wb") as file:
        if jsonl:
            file.write(encoder.encode_lines(data))
        else:
            file.write(encoder.encode(data))
    print(f"The file contains {len(data)} items.")
    print("Saved to", file_path)

def read_tsv(file_path, row_names=None):
    file_path = Path(file_path)
    if file_path.is_file() and file_path.suffix == ".tsv" :
        temp = pd.read_csv(file_path, sep='\t', names=row_names)
    else:
        raise ValueError("filepath is not a file or it is not a tsv file.")
    return temp


def parse_heideltime_result(heideltime_result, flatten=False):
    # root = etree.fromstring(heideltime_result, parser=parser)

    # values = []
    # for el in root.iter():
    #     if el.tag == "TIMEX3":
    #         value = el.attrib.get("value", "")
    #         if value and "2025" not in value and not value.startswith("P"):
    #             values.append({"text": el.text or "", "value": value})

    heideltime_result = heideltime_result.split("\n")[3]
    values = timex_text_pattern.findall(heideltime_result)

    # if flatten:
    #     return [field for item in values for field in (item["text"], item["value"])]

    return values


def parse_sutime_result(sutime_result, flatten=False):
    values = [x.get("text") for x in sutime_result if x.get("text")]
    # for x in sutime_result:
    #     text = x.get("text", "")
    #     # value = x.get("value", "")

    #     if isinstance(value, dict):
    #         value = ""
    #     elif not value or ("2025" not in value and value.startswith("P")):
    #         value = ""

    #     if flatten:
    #         values.append(text)
    #         if value:
    #             values.append(value)
    #     else:
    #         values.append({"text": text, "value": value})

    return values

# Read the corpus data

In [47]:
DATA_PATH = Path("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize")
SPLIT = "train"

CORPUS_TEMPORAL_JSONL_PATH = DATA_PATH / SPLIT / "chunked/corpus_temporal_refined.jsonl"
corpus_temporal_jsonl = read_json(CORPUS_TEMPORAL_JSONL_PATH, jsonl=True)

CHUNKED_CORPUS_JSONL_PATH = DATA_PATH / SPLIT / "chunked/corpus.jsonl"
chunked_corpus_jsonl = read_json(CHUNKED_CORPUS_JSONL_PATH, jsonl=True)

STITCHED_CORPUS_JSONL = []
for x, y in zip(chunked_corpus_jsonl, corpus_temporal_jsonl):
    x.update(y)
    STITCHED_CORPUS_JSONL.append(x)

The file is of type: <class 'list'>
The file contains 24804 items.
The file is of type: <class 'list'>
The file contains 24804 items.


In [48]:
texts = [x["text"] for x in STITCHED_CORPUS_JSONL]
llm_results = [x["temporal"] for x in STITCHED_CORPUS_JSONL]

In [ ]:
results = []

def init_worker():
    global sutime, heideltime_parser
    
    from sutime import SUTime
    sutime = SUTime(mark_time_ranges=True, include_range=True)

    from python_heideltime import Heideltime
    heideltime_parser = Heideltime()
    heideltime_parser.set_document_type("NEWS")


def get_temporal_set(text, llm_result, flatten=True):
    sutime_result = parse_sutime_result(sutime.parse(text), flatten=flatten)

    heideltime_result = parse_heideltime_result(
        heideltime_parser.parse(text), flatten=flatten
    )
    llm_result = llm_result.split(",")

    if flatten:
        return set(sutime_result).union(heideltime_result).union(llm_result)
    else:
        return sutime_result, heideltime_result, llm_result


def get_temporal_set_worker(args):
    text, llm_result = args
    return get_temporal_set(text, llm_result, flatten=True)

# if __name__ == "__main__":
#     with ThreadPoolExecutor(max_workers=16) as executor:
#         results = list(tqdm(
#             executor.map(get_temporal_set_worker, zip(texts, llm_results)),
#             total=len(texts)  # important to get progress bar length right
#         ))

In [ ]:
# write_json(str(DATA_PATH / SPLIT / "chunked/temporal_sutime_heideltime.jsonl"), results, jsonl=True)

The file contains 100 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/chunked/temporal_sutime_heideltime.jsonl


# Adding temporal expressions for each positive and negative passage in the training jsonl

In [63]:
DATA_PATH = Path(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize"
)
SPLIT = "train"

TRAIN_JSONL_PATH = DATA_PATH / SPLIT / "train.jsonl"
train_jsonl = read_json(TRAIN_JSONL_PATH, jsonl=True)

The file is of type: <class 'list'>
The file contains 8060 items.


In [ ]:
for ix, line in tqdm(enumerate(train_jsonl), total=len(train_jsonl)):
    positive_passages, negative_passages = line["positive_passages"], line["negative_passages"]

    for item in positive_passages:
        pos_docid = item["docid"]
        pos_text = item["text"]

        sutime_result = sutime.parse(pos_text)
        sutime_result = parse_sutime_result(sutime_result)

        heideltime_result = heideltime_parser.parse(pos_text)
        heideltime_result = parse_heideltime_result(heideltime_result)

        item['temporal'] = list(set(sutime_result + heideltime_result))

    for item in negative_passages:
        neg_docid = item["docid"]
        neg_text = item["text"]

        sutime_result = sutime.parse(neg_text)
        sutime_result = parse_sutime_result(sutime_result)

        heideltime_result = heideltime_parser.parse(neg_text)
        heideltime_result = parse_heideltime_result(heideltime_result)

        item["temporal"] = list(set(sutime_result + heideltime_result))

In [ ]:
# write_json(
#     "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal.jsonl",
#     train_jsonl,
#     jsonl=True
# )

The file contains 8060 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal.jsonl


# Dev TsRetriever with temporal

In [68]:
temp = {"docid":4161,"text":"Roberto Brown played for which team from 2005 to 2006?","temporal":["2005","2006","from 2005 to 2006"]}

In [70]:
docid = temp["docid"]
text = temp["text"]
temporal = temp["temporal"]

In [3]:
from datasets import load_dataset

ds = load_dataset(
    "zeta-alpha-ai/NanoNQ",
    "qrels",
    cache_dir="/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/tevatron",
)

Generating train split: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [00:00<00:00, 16597.84 examples/s]


In [7]:
ds['train']

Dataset({
    features: ['query-id', 'corpus-id'],
    num_rows: 57
})

In [9]:
for i in ds['train']:
    print(i)
    break

{'query-id': 'test1618', 'corpus-id': 'doc57226'}


In [12]:
with open("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/tevatron/zeta-alpha-ai___nano_nq/qrels.txt", "w") as f:
    for i in ds['train']:
        query_id, corpus_id = i['query-id'], i['corpus-id']
        f.write(f"{query_id} 0 {corpus_id} 1\n")